In [48]:
from dataclasses import dataclass
from pathlib import Path
from itertools import islice
from PIL import Image
import torch
import torch.nn as nn
from torchvision.transforms import v2
from ultralytics import YOLO

import mlflow
import mlflow.pytorch

TEST_DATASET_ROOT = Path.home() / Path("BLUEBERRY_DATA/datasets/dataset-c834a7b7")
IMAGES_DIR = TEST_DATASET_ROOT / "images"
LABELS_DIR = TEST_DATASET_ROOT / "annotations" / "obj_train_data"
IOU_THRESHOLD = 0.5
DEVICE = "cuda:0"

detector = YOLO("yolo_vs_classifier_v1.pt")
detector.to(DEVICE)

# ==========================================================
# LOAD classifier model
tracking_uri = Path("../experiments/mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{tracking_uri}")
run_id = "531b469fcd5f49ca86d611789277b437"
classifier = mlflow.pytorch.load_model(f"runs:/{run_id}/best_model")
classifier.eval()
# ==========================================================

inference_transform = v2.Compose([
  v2.ToImage(),
  v2.Resize((64,64)),
  v2.ToDtype(torch.float32, scale=True),
  # v2.Normalize(mean=, std=) # depends on which model i choose , what dataset it was trained
])

@dataclass
class BBox:
  cls: int # index
  xc: float # normalized
  yc: float
  w: float
  h: float

  def to_xyxy(self, img_h, img_w) -> tuple[int, int, int, int]:
    x1 = (self.xc - self.w/2) * img_w # x_min
    y1 = (self.yc - self.h/2) * img_h # y_min
    x2 = (self.xc + self.w/2) * img_w # x_max
    y2 = (self.yc + self.h/2) * img_h # y_max
    return int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))


def load_gt(label_file, img_h, img_w):
  '''
  gt bbox = (cls, [x1,y1,x2,y2])
  '''
  gt_boxes = []
  with label_file.open(encoding='utf-8') as f:
    for line in f:
      cls, xc, yc, w, h = map(float, line.split())
      box = BBox(int(cls),xc,yc,w,h)
      x1, y1, x2, y2 = box.to_xyxy(img_h, img_w)
      gt_boxes.append((int(cls), [x1, y1, x2, y2]))

  return gt_boxes

def iou(boxA, boxB):
  xA1, yA1, xA2, yA2 = boxA
  xB1, yB1, xB2, yB2 = boxB

  inter_x1 = max(xA1, xB1)
  inter_y1 = max(yA1, yB1)
  inter_x2 = min(xA2, xB2)
  inter_y2 = min(yA2, yB2)

  inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

  areaA = (xA2 - xA1) * (yA2 - yA1)
  areaB = (xB2 - xB1) * (yB2 - yB1)

  # union_area = areaA + areaB - inter_area
  return inter_area / (areaA + areaB - inter_area + 1e-6)


In [ ]:
'''
SOS: Make sure the labels in the dataset I use for validation correspond to the global labels i have 0:bud, 1:flower , etc...
'''

outputs = []
img_h, img_w = 2048, 1536
for label_file in islice(LABELS_DIR.iterdir(), 1):

  gt_boxes = load_gt(label_file, img_h, img_w)
  img_file = IMAGES_DIR / label_file.with_suffix('.png').name
  # print(img_file)
  img_rgb = Image.open(img_file).convert("RGB")
  results = detector(img_rgb)
  result = results[0]
  pred_boxes = result.boxes.xyxy.cpu().numpy() # (num_dets, 4)
  pred_cls = result.boxes.cls.cpu().numpy() # (num_dets,)
  # print(pred_boxes)

  # for each gt box find the prediction that corresponds to it
  outputs_per_file = []
  already_matched_pred_boxes = set()
  for gt_cls, gt_box in gt_boxes:
    # print(f"ground truth box: {gt_box}")
    best_iou, best_j = 0, -1

    for j, pbox in enumerate(pred_boxes):
      if j in already_matched_pred_boxes:
        continue

      score = iou(gt_box, pbox)
      if score > best_iou:
        best_iou = score
        best_j = j
        # print(f"best {j}, iou score: {score}")

    # past that poing best_j contains the best matching predicted bbox to the gt
    if best_iou < IOU_THRESHOLD:
      # missed prediction, false negative?
      outputs_per_file.append({
        "gt": gt_cls,
        "yolo": None,
        "classifier_pred": None,
        "classifier_conf": None,
        "match": False,
        "iou": None
      })
      # print("didn't match to any detection")
      continue

    already_matched_pred_boxes.add(best_j)
    # print(f"==================================")

    # YOLO class prediction
    yolo_pred = int(pred_cls[best_j])

    # TODO: feed the ground truth to the classifier
    # -----------------------------
    # Crop GT box for classifier
    # -----------------------------
    # x1, y1, x2, y2 =  gt_box
    crop = img_rgb.crop(gt_box)
    if crop.size == 0:
      continue

    crop_transformed = inference_transform(crop).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
      logits = classifier(crop_transformed).cpu()
    classifier_probs = nn.Softmax(dim=1)(logits).cpu() # (1, n_classes)
    classifier_pred = logits.argmax(dim=1).item()
    classifier_conf = classifier_probs[:, classifier_pred].item()
    # print(logits)
    # print(classifier_probs)
    # print(classifier_pred)
    # print(100*classifier_conf)

    outputs_per_file.append({
      "gt": gt_cls,
      "yolo": yolo_pred,
      "classifier_pred": classifier_pred,
      "classifier_conf": 100*classifier_conf,
      "match": True,
      "iou": best_iou
    })

  outputs.extend(outputs_per_file)



0: 512x384 190 greens, 1 pink, 11 blues, 7.1ms
Speed: 1.1ms preprocess, 7.1ms inference, 0.6ms postprocess per image at shape (1, 3, 512, 384)

0: 512x384 239 greens, 9 pinks, 29 blues, 6.5ms
Speed: 0.9ms preprocess, 6.5ms inference, 0.6ms postprocess per image at shape (1, 3, 512, 384)


In [62]:
len(outputs_per_file), len(outputs)

y_true = []
y_yolo = []
y_classifier = []
for d in outputs_per_file:
  if d['match']:

    y_true.append(d['gt'])
    y_yolo.append(d['yolo'])
    y_classifier.append(d['classifier_pred'])


print(y_true)
  

[1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 3, 2, 1, 1, 1, 3, 3, 3, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 2, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1]


589